In [2]:
!pip install requests pandas pyarrow

In [15]:
import requests
import pandas as pd
from datetime import datetime
import pyarrow

In [17]:
BASE_URL = "https://opendata.paris.fr/api/records/1.0/search/"

In [18]:
params = {
    "dataset": "velib-disponibilite-en-temps-reel",
    "rows": 10000,  # max pour exploration
}

response = requests.get(BASE_URL, params=params)
data = response.json()

In [19]:
records = data["records"]

rows = []
for r in records:
    row = r["fields"]
    row["record_timestamp"] = r["record_timestamp"]
    rows.append(row)

df = pd.DataFrame(rows)

In [20]:
df.head()

,name,stationcode,ebike,mechanical,coordonnees_geo,duedate,numbikesavailable,numdocksavailable,capacity,is_renting,is_installed,nom_arrondissement_communes,is_returning,code_insee_commune,record_timestamp
0,Hôpital Mondor,40001,7,5,"[48.798922410229, 2.4537451531298]",2026-02-09T16:08:34+00:00,12,16,28,OUI,OUI,Créteil,OUI,94028,2026-02-09T16:26:00.354Z
1,Toudouze - Clauzel,9020,3,0,"[48.87929591733507, 2.3373600840568547]",2026-02-09T16:07:50+00:00,3,17,21,OUI,OUI,Paris,OUI,75056,2026-02-09T16:26:00.354Z
2,Charonne - Robert et Sonia Delaunay,11104,1,1,"[48.855907555969, 2.3925706744194]",2026-02-09T16:06:19+00:00,2,18,20,OUI,OUI,Paris,OUI,75056,2026-02-09T16:26:00.354Z
3,Saint-Sulpice,6003,5,15,"[48.85165383178419, 2.3308077827095985]",2026-02-09T16:07:56+00:00,20,1,21,OUI,OUI,Paris,OUI,75056,2026-02-09T16:26:00.354Z
4,Vaneau - Sèvres,7002,5,4,"[48.848563233059, 2.3204218259346]",2026-02-09T16:08:59+00:00,9,24,35,OUI,OUI,Paris,OUI,75056,2026-02-09T16:26:00.354Z


In [21]:
df.columns

Index(['name', 'stationcode', 'ebike', 'mechanical', 'coordonnees_geo',
       'duedate', 'numbikesavailable', 'numdocksavailable', 'capacity',
       'is_renting', 'is_installed', 'nom_arrondissement_communes',
       'is_returning', 'code_insee_commune', 'record_timestamp'],
      dtype='str')

In [22]:
df["record_timestamp"] = pd.to_datetime(df["record_timestamp"])

df["availability_ratio"] = df["numbikesavailable"] / df["capacity"]

df[[
    "stationcode",
    "name",
    "nom_arrondissement_communes",
    "capacity",
    "numbikesavailable",
    "numdocksavailable",
    "availability_ratio",
    "record_timestamp"
]].head()

,stationcode,name,nom_arrondissement_communes,capacity,numbikesavailable,numdocksavailable,availability_ratio,record_timestamp
0,40001,Hôpital Mondor,Créteil,28,12,16,0.428571,2026-02-09 16:26:00.354000+00:00
1,9020,Toudouze - Clauzel,Paris,21,3,17,0.142857,2026-02-09 16:26:00.354000+00:00
2,11104,Charonne - Robert et Sonia Delaunay,Paris,20,2,18,0.100000,2026-02-09 16:26:00.354000+00:00
3,6003,Saint-Sulpice,Paris,21,20,1,0.952381,2026-02-09 16:26:00.354000+00:00
4,7002,Vaneau - Sèvres,Paris,35,9,24,0.257143,2026-02-09 16:26:00.354000+00:00


In [23]:
df.sort_values("availability_ratio").head(10)[[
    "name", "nom_arrondissement_communes", "capacity",
    "numbikesavailable", "availability_ratio"
]]

,name,nom_arrondissement_communes,capacity,numbikesavailable,availability_ratio
1494,Louis Dain - Rosiers,Saint-Ouen-sur-Seine,1,0,0.0
1491,Buisson Saint-Louis - Saint-Maur,Paris,30,0,0.0
19,Moulin de Pierre - Abbé Grégoire,Issy-les-Moulineaux,20,0,0.0
16,Le Brix et Mesmin - Jourdan,Paris,21,0,0.0
48,Lieutenant Colonel Prudhon - Repos,Argenteuil,26,0,0.0
1322,Mûriers - Père Lachaise,Paris,20,0,0.0
1326,Place Charles de Gaulle,Ville-d'Avray,24,0,0.0
1324,Chastenet de Géry - Marcel Paul,Villejuif,25,0,0.0
1075,Versailles - Résistances,Thiais,1,0,0.0
1050,Vivienne - Petits Champs,Paris,30,0,0.0


In [24]:
df.groupby("nom_arrondissement_communes").agg({
    "numbikesavailable": "sum",
    "capacity": "sum"
}).assign(
    availability_ratio=lambda x: x["numbikesavailable"] / x["capacity"]
).sort_values("availability_ratio")

,numbikesavailable,capacity,availability_ratio
nom_arrondissement_communes,,,
Ville-d'Avray,0,24,0.000000
Les Lilas,4,141,0.028369
Romainville,7,125,0.056000
Malakoff,19,238,0.079832
Le Pré-Saint-Gervais,4,42,0.095238
...,...,...,...
Levallois-Perret,202,360,0.561111
Orly,38,65,0.584615
Noisy-le-Sec,35,58,0.603448


In [25]:
df["hour"] = df["record_timestamp"].dt.hour

df.groupby("hour")["availability_ratio"].mean()

hour
16    0.372451
Name: availability_ratio, dtype: float64

In [26]:
df.to_parquet("velib_raw_snapshot.parquet", index=False)

ArrowKeyError: A type extension with name pandas.period already defined

In [19]:
import requests
import pandas as pd
from datetime import datetime, timezone
import os

API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/records"

now = datetime.now(timezone.utc)
date = now.strftime("%Y-%m-%d")
hour = now.strftime("%H")
ts = now.strftime("%Y%m%d_%H%M%S")

params = {"limit": 10}
r = requests.get(API_URL, params=params)

print("Status:", r.status_code)
print("\nKeys:", r.json().keys())
print("\nFull response:")
print(r.json())

data = r.json()["results"]

df = pd.DataFrame(data)
df["ingestion_timestamp"] = now.isoformat()
df["snapshot_id"] = ts

base_path = f"/app/data_lake/bronze/velib/ingestion_date={date}/hour={hour}"
os.makedirs(base_path, exist_ok=True)

file_path = f"{base_path}/snapshot_{ts}.parquet"
df.to_parquet(file_path, index=False)

print(f"Snapshot saved: {file_path}")



Status: 200

Keys: dict_keys(['total_count', 'results'])

Full response:
{'total_count': 1506, 'results': [{'stationcode': '16107', 'name': 'Benjamin Godard - Victor Hugo', 'is_installed': 'OUI', 'capacity': 35, 'numdocksavailable': 14, 'numbikesavailable': 21, 'mechanical': 8, 'ebike': 13, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:11:22+00:00', 'coordonnees_geo': {'lon': 2.275725, 'lat': 48.865983}, 'nom_arrondissement_communes': 'Paris', 'code_insee_commune': '75056', 'station_opening_hours': None}, {'stationcode': '40001', 'name': 'Hôpital Mondor', 'is_installed': 'OUI', 'capacity': 28, 'numdocksavailable': 16, 'numbikesavailable': 11, 'mechanical': 6, 'ebike': 5, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:10:25+00:00', 'coordonnees_geo': {'lon': 2.4537451531298, 'lat': 48.798922410229}, 'nom_arrondissement_communes': 'Créteil', 'code_insee_commune': '94028', 'station_opening_hours': None}, {'stationcode': '32304', 'name': 'Char

PermissionError: [Errno 13] Permission denied: '/app'

In [21]:
import requests
import pandas as pd

API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/records"

# Test avec limite basse
params = {"limit": 100}
r = requests.get(API_URL, params=params)

# Afficher la structure
print("Status:", r.status_code)
print("\nKeys:", r.json().keys())
print("\nFull response:")
print(r.json())

# Si "results" existe
if "results" in r.json():
    df = pd.DataFrame(r.json()["results"])
    print("\n✅ DataFrame créé avec 'results'")
    print(df.head())
elif "records" in r.json():
    df = pd.DataFrame(r.json()["records"])
    print("\n✅ DataFrame créé avec 'records'")
    print(df.head())

Status: 200

Keys: dict_keys(['total_count', 'results'])

Full response:
{'total_count': 1506, 'results': [{'stationcode': '16107', 'name': 'Benjamin Godard - Victor Hugo', 'is_installed': 'OUI', 'capacity': 35, 'numdocksavailable': 14, 'numbikesavailable': 21, 'mechanical': 8, 'ebike': 13, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:11:22+00:00', 'coordonnees_geo': {'lon': 2.275725, 'lat': 48.865983}, 'nom_arrondissement_communes': 'Paris', 'code_insee_commune': '75056', 'station_opening_hours': None}, {'stationcode': '40001', 'name': 'Hôpital Mondor', 'is_installed': 'OUI', 'capacity': 28, 'numdocksavailable': 16, 'numbikesavailable': 11, 'mechanical': 6, 'ebike': 5, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:10:25+00:00', 'coordonnees_geo': {'lon': 2.4537451531298, 'lat': 48.798922410229}, 'nom_arrondissement_communes': 'Créteil', 'code_insee_commune': '94028', 'station_opening_hours': None}, {'stationcode': '32304', 'name': 'Char

Status: 200

Keys: dict_keys(['total_count', 'results'])

Full response:
{'total_count': 1506, 'results': [{'stationcode': '16107', 'name': 'Benjamin Godard - Victor Hugo', 'is_installed': 'OUI', 'capacity': 35, 'numdocksavailable': 14, 'numbikesavailable': 21, 'mechanical': 8, 'ebike': 13, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:11:22+00:00', 'coordonnees_geo': {'lon': 2.275725, 'lat': 48.865983}, 'nom_arrondissement_communes': 'Paris', 'code_insee_commune': '75056', 'station_opening_hours': None}, {'stationcode': '40001', 'name': 'Hôpital Mondor', 'is_installed': 'OUI', 'capacity': 28, 'numdocksavailable': 16, 'numbikesavailable': 11, 'mechanical': 6, 'ebike': 5, 'is_renting': 'OUI', 'is_returning': 'OUI', 'duedate': '2026-02-10T22:10:25+00:00', 'coordonnees_geo': {'lon': 2.4537451531298, 'lat': 48.798922410229}, 'nom_arrondissement_communes': 'Créteil', 'code_insee_commune': '94028', 'station_opening_hours': None}, {'stationcode': '32304', 'name': 'Char